# Task 2 Model Experimentation and Development - Global AI & Data Jobs Salary Dataset

## Objective

The purpose of this modelling task is to develop and evaluate a machine learning regression pipeline using PyCaret to predict the annual base salary of AI and data-related jobs.

The primary target variable is `salary_usd`. The modelling process will include preprocessing and transformation, model comparison using k-fold cross-validation, hyperparameter tuning, evaluation using multiple regression performance metrics, and selection of the best-performing model.

MLflow will also be used to track model experiments, metrics and artifacts throughout the model development process.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from hydra import compose, initialize

PROJECT_ROOT = Path.cwd().parents[1]

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name="config")

raw_data_path = PROJECT_ROOT / cfg.data.raw_path

df = pd.read_csv(raw_data_path)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (90000, 35)


,id,country,job_role,ai_specialization,experience_level,experience_years,salary_usd,bonus_usd,education_required,industry,...,vacation_days,skill_demand_score,automation_risk,job_security_score,career_growth_score,work_life_balance_score,promotion_speed,salary_percentile,cost_of_living_index,employee_satisfaction
0,1,UAE,Machine Learning Engineer,Reinforcement Learning,Entry,0,66465,5395,Master,Automotive,...,27,12,76,57,65,73,15,55,1.23,76
1,2,USA,AI Engineer,LLM,Entry,1,75507,11713,Bootcamp,Retail,...,27,54,29,69,60,51,15,58,0.87,67
2,3,Brazil,Research Scientist,Analytics,Entry,0,41660,5268,PhD,Healthcare,...,13,12,49,70,59,68,37,13,2.13,61
3,4,India,Software Engineer AI,Computer Vision,Senior,6,43268,7975,Diploma,Tech,...,30,80,47,79,65,55,46,74,1.49,56
4,5,Germany,Machine Learning Engineer,Computer Vision,Entry,0,69119,4758,Master,Retail,...,24,82,47,64,52,69,17,21,0.87,72


## 1. Modelling Preparation

The target variable for this regression task is `salary_usd`. The `id` column is a unique record identifier and does not contain meaningful predictive information, therefore it will be excluded from model training.

Multiple controlled experiments will be conducted to evaluate the effect of different modelling decisions. The initial experiment will use all candidate predictors with the original target variable to establish a baseline. Subsequent experiments will investigate automated feature selection and target transformation.

As the EDA identified a positively skewed distribution for `salary_usd`, a log-transformed target will also be evaluated to determine whether reducing target skew improves model performance.

In [3]:
target = cfg.model.target

ignore_features = ["id"]

print(f"Target variable: {target}")
print(f"Features excluded from modelling: {ignore_features}")
print(f"Dataset shape before PyCaret setup: {df.shape}")

Target variable: salary_usd
Features excluded from modelling: ['id']
Dataset shape before PyCaret setup: (90000, 35)


## 2. Experiment 1 - Baseline Models with All Candidate Features

The first experiment retains all candidate predictors except the unique identifier `id` and uses the original `salary_usd` target without feature selection or target transformation.

This experiment establishes a baseline against which subsequent feature-selection and target-transformation experiments can be compared.

### 2.1 Initialise Baseline PyCaret Experiment

The baseline PyCaret regression experiment is initialised using an 80/20 train-test split and 10-fold cross-validation.

All candidate predictors are retained except `id`. No automated feature selection or target transformation is applied in this experiment. A fixed random state is used to ensure that the experiment is reproducible.

PyCaret automatically handles the required categorical encoding and constructs the preprocessing pipeline before model training.

In [ ]:
import os
import mlflow

os.environ["GIT_PYTHON_REFRESH"] = "quiet" # does not complain about GIT login 
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("salary_baseline_all_features")


<Experiment: artifact_location='mlflow-artifacts:/202828570265027712', creation_time=1787221909205, experiment_id='202828570265027712', last_update_time=1787221909205, lifecycle_stage='active', name='salary_baseline_all_features', tags={}>

In [5]:
from pycaret.regression import RegressionExperiment

baseline_exp = RegressionExperiment()

baseline_exp.setup(
    data=df,
    target=target,
    ignore_features=ignore_features,
    train_size=1 - cfg.training.test_size,
    fold_strategy="kfold",
    fold=cfg.training.fold,
    fold_shuffle=True,
    session_id=cfg.model.random_state,
    log_experiment="mlflow",
    experiment_name="salary_baseline_all_features",
    verbose=True
)

,Description,Value
0,Session id,42
1,Target,salary_usd
2,Target type,Regression
3,Original data shape,"(90000, 35)"
4,Transformed data shape,"(90000, 81)"
5,Transformed train set shape,"(72000, 81)"
6,Transformed test set shape,"(18000, 81)"
7,Ignore features,1
8,Numeric features,25
9,Categorical features,8
